# Day 4 student exercise: repair a multilingual measure

**Time:** about 20–25 minutes. This notebook is self-contained.
Run the prepared setup, then edit only the three marked cells.
Keep the same 36 constructed examples and human labels throughout.


## 1. Prepared setup — run without editing


In [ ]:
from pathlib import Path
import copy
import re
import pandas as pd

def find_data_dir():
    for path in [Path("materials/data"), Path("../data"), Path("data")]:
        if (path / "multilingual_climate_examples.csv").exists():
            return path
    raise FileNotFoundError("Start JupyterLab in the course root or materials/notebooks.")

DATA_DIR = find_data_dir()
texts = pd.read_csv(DATA_DIR / "multilingual_climate_examples.csv")
assert texts["id"].is_unique
assert set(texts["lang"]) == {"en", "de", "es"}
texts.head()


In [ ]:
def tokenize(text):
    # Unicode letters, with punctuation and numbers separated.
    return re.findall(r"[^\W\d_]+", str(text).casefold(), flags=re.UNICODE)

# Small baseline dictionaries. The omissions give us errors to inspect.
LEXICONS = {
    "en": {
        "policy": [r"tax\w*", r"rule\w*", r"subsid\w*", r"ban\w*", r"law\w*"],
        "protest": [r"activist\w*", r"march\w*", r"rally\w*", r"protest\w*",
                    r"block\w*", r"demonstrat\w*"],
        "impact": [r"flood\w*", r"drought\w*", r"heatwave\w*", r"wildfire\w*", r"fire\w*"],
    },
    "de": {
        "policy": [r"steuer\w*", r"gesetz\w*", r"regel\w*", r"zusch\w*", r"verbot\w*"],
        "protest": [r"aktivist\w*", r"demonstr\w*", r"kundgebung\w*",
                    r"protest\w*", r"blockier\w*"],
        "impact": [r"überschwemm\w*", r"dürre\w*", r"hitzewell\w*",
                   r"waldbränd\w*", r"trockenheit\w*"],
    },
    "es": {
        "policy": [r"impuesto\w*", r"ley\w*", r"norma\w*", r"subsidio\w*", r"prohibic\w*"],
        "protest": [r"activista\w*", r"march\w*", r"manifest\w*",
                    r"protest\w*", r"bloque\w*"],
        "impact": [r"inund\w*", r"sequía\w*", r"calor\w*", r"incendio\w*", r"ola\w*"],
    },
}

def dictionary_scores(text, language, lexicons=LEXICONS):
    tokens = tokenize(text)
    return {
        label: sum(any(re.fullmatch(pattern, token) for pattern in patterns)
                   for token in tokens)
        for label, patterns in lexicons[language].items()
    }

def dictionary_label(text, language, lexicons=LEXICONS):
    scores = dictionary_scores(text, language, lexicons)
    top = max(scores.values())
    winners = [label for label, value in scores.items() if value == top]
    return winners[0] if top > 0 and len(winners) == 1 else "uncoded"

def predict_original(frame, lexicons=LEXICONS):
    return [dictionary_label(text, lang, lexicons)
            for text, lang in zip(frame["text"], frame["lang"])]

def predict_translated(frame, english=None):
    lexicons = {"en": english if english is not None else LEXICONS["en"]}
    return [dictionary_label(text, "en", lexicons) for text in frame["translation_en"]]


In [ ]:
def compare_routes(frame, original, translated):
    left = frame.assign(method="Original dictionaries", predicted=original)
    right = frame.assign(method="English translations", predicted=translated)
    results = pd.concat([left, right], ignore_index=True)
    results["correct"] = results["predicted"].eq(results["gold"])
    results["coded"] = results["predicted"].ne("uncoded")
    return results

def language_summary(results):
    return (results.groupby(["method", "lang"], sort=False)
            .agg(n=("id", "size"), coverage=("coded", "mean"),
                 accuracy=("correct", "mean")).reset_index())


In [ ]:
baseline_original = predict_original(texts)
baseline_translated = predict_translated(texts)
baseline_results = compare_routes(texts, baseline_original, baseline_translated)
language_summary(baseline_results)


## 2. STUDENT EDIT 1 — repair a German pattern

The pattern for regel misses the compound Emissionsregeln in de02.
Add one German policy pattern that matches the whole token.


In [ ]:
student_lexicons = copy.deepcopy(LEXICONS)
# STUDENT EDIT 1: replace None with one regular expression string.
german_policy_pattern = None
assert isinstance(german_policy_pattern, str), "Add one German pattern."
student_lexicons["de"]["policy"].append(german_policy_pattern)
repaired_original = predict_original(texts, student_lexicons)
row = texts.index[texts["id"].eq("de02")][0]
print("Before:", baseline_original[row], "After:", repaired_original[row])


## 3. STUDENT EDIT 2 — repair an English pattern

The English paraphrase of de11 says extreme heat. Add one pattern
to the English impact category. Think about false positives.


In [ ]:
student_english = copy.deepcopy(LEXICONS["en"])
# STUDENT EDIT 2: replace None with one regular expression string.
english_impact_pattern = None
assert isinstance(english_impact_pattern, str), "Add one English pattern."
student_english["impact"].append(english_impact_pattern)
repaired_translated = predict_translated(texts, student_english)
row = texts.index[texts["id"].eq("de11")][0]
print("Before:", baseline_translated[row], "After:", repaired_translated[row])


## 4. STUDENT EDIT 3 — compare by language

Coverage is the share that received a category. Accuracy is the share
matching the human label, with uncoded treated as an error.


In [ ]:
student_results = compare_routes(texts, repaired_original, repaired_translated)
rows = []
for (method, lang), group in student_results.groupby(["method", "lang"]):
    # STUDENT EDIT 3: replace both None values with expressions using group.
    coverage = None
    accuracy = None
    assert coverage is not None and accuracy is not None
    rows.append({"method": method, "lang": lang, "n": len(group),
                 "coverage": coverage, "accuracy": accuracy})
student_summary = pd.DataFrame(rows)
student_summary


## 5. Inspect one remaining error

Explain whether it comes from a missing expression, a misleading
match, a translation choice, or an unclear category.


In [ ]:
remaining = student_results.loc[
    ~student_results["correct"],
    ["method", "id", "lang", "text", "translation_en", "gold", "predicted"]]
remaining.sort_values(["lang", "id", "method"])


**Write two sentences:** What improved? Which error would you investigate first in a real study?
